In [14]:
#Importing libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

#Loading and cleaning the dataset

df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip()

df["album_name"] = (
    df["album_name"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

def clean_artist(x):
    if pd.isna(x):
        return "Unknown Artist"
    return str(x).split(";")[0].strip()

if "artists" in df.columns:
    df["clean_artist"] = df["artists"].apply(clean_artist)
    album_meta = df.groupby("album_name")["clean_artist"].first()
else:
    df["clean_artist"] = "Unknown Artist"
    album_meta = pd.Series("Unknown Artist", index=df["album_name"].unique())

if "genre" in df.columns:
    df["genre"] = df["genre"].fillna("unknown").str.lower().str.strip()
    album_genres = df.groupby("album_name")["genre"].first().to_dict()
else:
    album_genres = {}

true_counts = df.groupby(["clean_artist", "album_name"]).size().reset_index(name="actual_track_count")
album_counts = dict(zip(true_counts["album_name"], true_counts["actual_track_count"]))

features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness",
    "valence", "tempo"
]
features = [f for f in features if f in df.columns]

album_df = df.groupby("album_name")[features].mean()

if "tempo" in album_df.columns:
    album_df["tempo"] = album_df["tempo"].replace(0, np.nan)
    album_df["tempo"] = album_df["tempo"].fillna(album_df["tempo"].median())

scaler = StandardScaler()
album_df_scaled = album_df.copy()
album_df_scaled[features] = scaler.fit_transform(album_df[features])

X = album_df_scaled[features].values

def is_low_quality(album):
    tc = album_counts.get(album, 0)
    return tc < 5

def search_albums(query):
    return album_df.index[
        album_df.index.str.contains(query, case=False, na=False)
    ].tolist()

def select_album(query):
    matches = search_albums(query)

    if len(matches) == 0:
        return None

    if len(matches) == 1:
        name = matches[0]
        print(f"\nAuto-selected: {name} — {album_meta.get(name, 'Unknown Artist')}")
        return name

    print("\nSelect an album:\n")
    print(f"{'No.':<4}{'Album':<45}{'Artist'}")
    print("-" * 80)

    for i, name in enumerate(matches):
        artist = album_meta.get(name, "Unknown Artist")
        short = name[:42] + "..." if len(name) > 42 else name
        print(f"{i+1:<4}{short:<45}{artist}")

    choice = int(input("\nEnter number: ")) - 1
    return matches[choice]

#Function for recommendations

def recommend(album_name, top_n=10):
    idx = album_df.index.get_loc(album_name)

    similarities = cosine_similarity(
        X[idx].reshape(1, -1),
        X
    )[0]

    sorted_idx = similarities.argsort()[::-1]
    target_genre = album_genres.get(album_name, "unknown")

    print("\nRecommendations:\n")
    print(f"{'Rank':<5}{'Album':<45}{'Artist':<25}{'Score'}")
    print("-" * 95)

    rank = 1

    for pos in sorted_idx:
        name = album_df.index[pos]

        if name == album_name:
            continue

        if is_low_quality(name):
            continue

        if album_genres:
            current_genre = album_genres.get(name, "unknown")

            if target_genre != "unknown" and current_genre != "unknown":
                target_words = set(target_genre.replace("-", " ").split())
                current_words = set(current_genre.replace("-", " ").split())

                if not target_words.intersection(current_words):
                    continue

        artist = album_meta.get(name, "Unknown Artist")
        score = similarities[pos]
        short = name[:42] + "..." if len(name) > 42 else name

        print(f"{rank:<5}{short:<45}{artist:<25}{score:.3f}")

        rank += 1
        if rank > top_n:
            break

query = input("Search album: ")
selected = select_album(query)

if selected:
    recommend(selected)
else:
    print("No album found")

Search album: Self titled

Auto-selected: Self Titled 1998 — Dropdead

Recommendations:

Rank Album                                        Artist                   Score
-----------------------------------------------------------------------------------------------
1    A Taste of Extreme Divinity                  Hypocrisy                0.991
2    Full of Hell & Merzbow                       Full Of Hell             0.987
3    The Inalienable Dreamless                    Discordance Axis         0.987
4    Mental Hygiene                               Internal Rot             0.981
5    Devious Persecution and Wholesale Slaughte...P.L.F.                   0.981
6    Grieving Birth                               Internal Rot             0.980
7    Disentanglement                              Fluoride                 0.979
8    Human Overdose                               Hatred Surge             0.977
9    Born in Shroud, Dead in Womb                 Ekwon                    0.974
10   